# 🔬 MammoAI — Detección de Cáncer de Mama

> **Sistema Open Source** de IA para detección, localización y análisis cuantitativo de mamografías.
> Arquitecturas: EfficientNet · ConvNeXt · ViT · Faster R-CNN · DETR
> Dataset: CBIS-DDSM

---

## 📋 Instrucciones previas

Antes de ejecutar el notebook, configura tus secrets en Colab:

1. En el menú de la izquierda, haz clic en el ícono 🔑 **(Secrets)**
2. Agrega los siguientes secrets:
   - **`GITHUB_TOKEN`** → Tu Personal Access Token de GitHub (con permisos `repo`)
   - **`HF_TOKEN`** → (Opcional) Tu token de HuggingFace para descargar modelos privados

> ⚠️ **NUNCA** escribas tu token directamente en el código.

---

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELDA 1 — Verificación de GPU y entorno                   ║
# ╚══════════════════════════════════════════════════════════════╝
import subprocess, sys, os

# Verificar GPU
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print('✅ GPU disponible:')
    # Extraer solo la info relevante
    lines = result.stdout.split('\n')
    for line in lines[8:12]:
        print(' ', line)
else:
    print('⚠️  No se detectó GPU. Ve a: Entorno de ejecución → Cambiar tipo de entorno → GPU')

# Verificar Python y CUDA
import torch
print(f'\n🐍 Python {sys.version.split()[0]}')
print(f'🔥 PyTorch {torch.__version__}')
print(f'💻 CUDA disponible: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'🎮 GPU: {torch.cuda.get_device_name(0)}')
    print(f'💾 VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# Montar Google Drive (opcional)
MOUNT_DRIVE = False  # Cambiar a True si quieres persistir datasets y modelos
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    print('✅ Google Drive montado.')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELDA 2 — Instalación de dependencias                     ║
# ╚══════════════════════════════════════════════════════════════╝
print('📦 Instalando dependencias...')
print('   (Este proceso tarda ~3-5 minutos en la primera ejecución)')

deps = [
    'torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118',
    'gradio plotly>=5.18.0',
    'timm>=0.9.12 opencv-python-headless pillow scikit-image',
    'grad-cam>=1.4.8',
    'pydicom>=2.4.3',
    'huggingface_hub>=0.20.3 datasets>=2.16.0 transformers>=4.37.0',
    'scikit-learn numpy',
    'onnx safetensors reportlab',
    'gitpython tqdm requests',
]

for dep in deps:
    print(f'  Installing: {dep.split()[0]}...')
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps'] + dep.split(),
        capture_output=True
    )

print('\n✅ Todas las dependencias instaladas.')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELDA 3 — Configuración de credenciales (Colab Secrets)   ║
# ╚══════════════════════════════════════════════════════════════╝
from google.colab import userdata

# Leer token de GitHub desde Colab Secrets
try:
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
    print('✅ GITHUB_TOKEN cargado desde Colab Secrets.')
except Exception:
    GITHUB_TOKEN = ''
    print('⚠️  GITHUB_TOKEN no encontrado en Secrets.')
    print('   Ve a 🔑 Secrets en el panel izquierdo y agrega GITHUB_TOKEN.')

# Token de HuggingFace (opcional)
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = HF_TOKEN
    print('✅ HF_TOKEN configurado.')
except Exception:
    HF_TOKEN = ''
    print('ℹ️  HF_TOKEN no configurado (opcional para modelos públicos).')

print(f'\n📋 Configuración:')
print(f'   GitHub Token: {"Configurado ✅" if GITHUB_TOKEN else "No configurado ⚠️"}')
print(f'   HF Token:     {"Configurado ✅" if HF_TOKEN     else "No configurado (OK para modelos públicos)"}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELDA 4 — Clonar / Actualizar repositorio GitHub          ║
# ╚══════════════════════════════════════════════════════════════╝
import os, subprocess, sys
from pathlib import Path

REPO_URL   = 'https://github.com/AderDevP/IA-Models'
LOCAL_DIR  = '/content/IA-Models'
BRANCH     = 'main'

if GITHUB_TOKEN:
    auth_url = REPO_URL.replace('https://', f'https://{GITHUB_TOKEN}@')
else:
    auth_url = REPO_URL  # Solo lectura sin token

if Path(LOCAL_DIR).exists():
    print(f'📁 Repositorio ya existe. Forzando sincronización con origin/{BRANCH}...')
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=LOCAL_DIR, capture_output=True)
    result = subprocess.run(
        ['git', 'reset', '--hard', f'origin/{BRANCH}'],
        cwd=LOCAL_DIR, capture_output=True, text=True
    )
    print(result.stdout or result.stderr)
else:
    print(f'⬇️  Clonando {REPO_URL}...')
    result = subprocess.run(
        ['git', 'clone', '--branch', BRANCH, auth_url, LOCAL_DIR],
        capture_output=True, text=True
    )
    safe_output = (result.stdout + result.stderr).replace(GITHUB_TOKEN or 'x', '***')
    print(safe_output)

# Agregar al path de Python
if LOCAL_DIR not in sys.path:
    sys.path.insert(0, LOCAL_DIR)

os.chdir(LOCAL_DIR)
print(f'\n✅ Directorio de trabajo: {os.getcwd()}')
print(f'📂 Archivos del proyecto:')
for f in sorted(Path(LOCAL_DIR).iterdir()):
    icon = '📁' if f.is_dir() else '📄'
    print(f'   {icon} {f.name}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELDA 5 — Descarga automática de modelos preentrenados    ║
# ╚══════════════════════════════════════════════════════════════╝
import importlib
import model_downloader
importlib.reload(model_downloader)
from model_downloader import ModelDownloader

downloader = ModelDownloader()

print('📋 Estado actual de modelos:')
print(downloader.status_report())

# Modelos a descargar automáticamente al inicio
# Modifica esta lista según tus necesidades
AUTO_DOWNLOAD_MODELS = [
    'efficientnet_b4_cbis',     # EfficientNet-B4 — clasificador principal
    'convnext_small_mammo',     # ConvNeXt-Small — alternativa moderna
    'fasterrcnn_resnet50_mammo', # Faster R-CNN — detector con bboxes
]

print('\n⬇️  Descargando modelos seleccionados...')
for model_id in AUTO_DOWNLOAD_MODELS:
    if not downloader.is_installed(model_id):
        print(f'\nDescargando: {model_id}...')
        try:
            path = downloader.download(model_id, progress_callback=print)
            print(f'✅ {model_id} → {path}')
        except Exception as e:
            print(f'❌ Error en {model_id}: {e}')
    else:
        print(f'✅ {model_id} ya instalado.')

print('\n📋 Estado final:')
print(downloader.status_report())

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELDA 6 — Lanzar el Dashboard MammoAI                     ║
# ╚══════════════════════════════════════════════════════════════╝
# Esta celda recarga el módulo app de forma forzada y lanza el servidor.

import importlib, sys
import app as mammo_app
importlib.reload(mammo_app)

print('🚀 Iniciando MammoAI Dashboard...')
print('   El enlace público aparecerá en unos segundos.')
print('   Puedes acceder desde cualquier dispositivo.')
print()

demo = mammo_app.build_app()
mammo_app.launch_app(demo, share=True)

---

## 🔧 Celdas de Utilidad (Opcionales)

Las siguientes celdas son herramientas adicionales que puedes ejecutar según necesites.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  UTILIDAD A — Descarga de subconjunto CBIS-DDSM (~10 GB)  ║
# ╚══════════════════════════════════════════════════════════════╝
# Ejecuta esta celda si quieres pre-descargar el dataset para entrenamiento.

from tasks.breast_cancer.dataset import CBISDDSMDataset

print('📥 Descargando subconjunto CBIS-DDSM (~10 GB)...')
print('   Esto puede tardar 15-30 minutos dependiendo de la velocidad de Colab.')
print()

# max_samples limita el número de imágenes para no exceder ~10 GB
dataset = CBISDDSMDataset(
    split='train',
    max_samples=5000,  # ~250 MB en PNG comprimido
)

print(f'\n✅ Dataset listo: {len(dataset)} muestras')
print(f'   Benignas:  {sum(1 for _,l in dataset.samples if l==0)}')
print(f'   Malignas:  {sum(1 for _,l in dataset.samples if l==1)}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  UTILIDAD B — Push manual a GitHub desde Colab             ║
# ╚══════════════════════════════════════════════════════════════╝
# Puedes usar la pestaña Git en el Dashboard, o ejecutar esto directamente.

from git_utils import GitManager

if not GITHUB_TOKEN:
    print('⚠️  Configura GITHUB_TOKEN en Colab Secrets primero.')
else:
    gm = GitManager(
        token=GITHUB_TOKEN,
        local_dir='/content/IA-Models',
    )
    
    # Ver estado del repo
    status = gm.get_status()
    print('📋 Estado del repositorio:')
    for k, v in status.items():
        print(f'   {k}: {v}')
    
    # Descomenta para hacer push:
    # success, output = gm.commit_and_push(
    #     add_all=True,
    #     message='feat: update from MammoAI Colab session',
    #     progress_callback=print,
    # )
    # print('Success:', success)

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  UTILIDAD C — Inferencia rápida desde línea de código      ║
# ╚══════════════════════════════════════════════════════════════╝
# Útil para pruebas rápidas sin abrir el Dashboard.

from PIL import Image
from core.registry import TaskRegistry
from detector import full_diagnostic_pipeline
import tasks  # auto-registro

# Cambiar por la ruta de tu imagen de prueba
TEST_IMAGE_PATH = '/content/test_mammogram.png'  

if not Path(TEST_IMAGE_PATH).exists():
    # Crear imagen de prueba sintética (solo para demostración)
    import numpy as np
    dummy = Image.fromarray(np.random.randint(50, 200, (512, 512), dtype=np.uint8)).convert('RGB')
    dummy.save(TEST_IMAGE_PATH)
    print(f'📸 Imagen de prueba sintética creada: {TEST_IMAGE_PATH}')

# Ejecutar diagnóstico
task    = TaskRegistry.load_task('breast_cancer')
image   = Image.open(TEST_IMAGE_PATH).convert('RGB')
device  = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'

annotated, heatmap, report = full_diagnostic_pipeline(
    image=image,
    model_id='efficientnet_b4_cbis',
    task=task,
    device=device,
)

print(report['report_text'])

# Mostrar imágenes en Colab
from IPython.display import display
display(annotated)
display(heatmap)